# Detecção de Defeitos em PCBs utilizando Faster R-CNN

>&nbsp;&nbsp;&nbsp;&nbsp;Alexandre Augusto Tescaro Oliveira

## 1. Introdução e Motivação

> **Resumo do Estudo:** Este trabalho apresenta o desenvolvimento de um sistema de Visão Computacional para detecção automática de defeitos em Placas de Circuito Impresso (PCBs), utilizando a arquitetura **Faster R-CNN**. Para viabilizar a execução local e manter comparação consistente entre modelos, o conjunto de dados anotado no formato COCO foi organizado diretamente sob os arquivos extraídos no *EDA*, preservando a distribuição das classes de defeito.

### 1.1 Contextualização
No cenário da Indústria 4.0, a garantia de qualidade na fabricação de componentes eletrônicos é crítica para reduzir perdas, retrabalho e falhas em campo. As Placas de Circuito Impresso (PCBs) são a base de praticamente todos os dispositivos eletrônicos modernos. Com a miniaturização dos componentes, a inspeção visual tornou-se mais complexa e exige soluções automáticas robustas.

### 1.2 O Problema
Tradicionalmente, a inspeção de PCBs é realizada de forma manual por operadores humanos ou por algoritmos de visão clássica baseados em regras rígidas. Esses métodos apresentam limitações no ambiente industrial:
>* **Fadiga Humana:** A inspeção visual repetitiva aumenta a chance de erro e inconsistências.
>* **Baixa Escalabilidade:** A inspeção manual é lenta e cria gargalos na linha de produção.
>* **Sensibilidade de Regras:** Métodos clássicos falham com variações de iluminação, rotação e ruído.

### 1.3 A Solução Proposta
Para reduzir esses problemas e automatizar o processo de inspeção, este notebook adota Deep Learning com **Faster R-CNN** (detector de duas etapas), buscando boa qualidade de localização de caixas em defeitos pequenos.

O objetivo é identificar e localizar seis tipos comuns de defeitos de fabricação:
>1.  **Missing Hole** (Furo faltante)
>2.  **Mouse Bite** (Mordida de rato/Falha na borda)
>3.  **Open Circuit** (Circuito aberto)
>4.  **Short** (Curto-circuito)
>5.  **Spur** (Esporão/Rebarba)
>6.  **Spurious Copper** (Cobre residual)


## 2. Análise Exploratória dos Dados (EDA)
> Notebook pode ser encontrado em ./EDA_VC.ipynb

> Link de acesso ao Dataset utilizado: https://www.kaggle.com/datasets/norbertelter/pcb-defect-dataset

> Para o Faster R-CNN foi necessário adaptar o dataset para o formato COCO, devido à incompatibilidade nativa com a notação de coordenadas do YOLO. Para essa etapa, ao invés de depender de plataformas de terceiros limitadas (como o Roboflow), foi utilizado o conversor customizado open-source **[Yolo-to-COCO-format-converter](https://github.com/alehholiveira/Yolo-to-COCO-format-converter)**. Este fork foi aprimorado para suportar o formato moderno de pastas do YOLO, garantindo que todo o subset gerado na EDA (5.000+ imagens) pudesse ser convertido dinamicamente para o padrão JSON lido pelo PyTorch.

In [ ]:
import importlib.util
import subprocess
import sys

FORCE_REINSTALL_TORCH = False
PREFER_CUDA_ON_NVIDIA = True
CUDA_INDEX_URL = "https://download.pytorch.org/whl/cu121"

def _run_cmd(args):
    print("$", " ".join(args))
    subprocess.check_call(args)

def _module_exists(module_name):
    return importlib.util.find_spec(module_name) is not None

def _has_nvidia_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "-L"],
            check=False,
            capture_output=True,
            text=True,
        )
        return result.returncode == 0 and bool(result.stdout.strip())
    except FileNotFoundError:
        return False

def _torch_stack_ready(needs_cuda):
    required_modules = ["torch", "torchvision", "torchaudio"]
    if not all(_module_exists(module_name) for module_name in required_modules):
        return False

    import torch

    if needs_cuda:
        return torch.cuda.is_available() and (torch.version.cuda is not None)
    return True

def _install_torch_stack(needs_cuda):
    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "torch",
        "torchvision",
        "torchaudio",
    ]
    if needs_cuda:
        install_cmd += ["--index-url", CUDA_INDEX_URL]

    try:
        _run_cmd(install_cmd)
    except subprocess.CalledProcessError:
        print("Primeira tentativa falhou. Limpando stack PyTorch e tentando novamente...")
        _run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"])
        _run_cmd(install_cmd)

nvidia_gpu_detected = _has_nvidia_gpu()
needs_cuda = PREFER_CUDA_ON_NVIDIA and nvidia_gpu_detected

# Garante ferramentas básicas de build/instalação no venv recém-criado.
_run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

if FORCE_REINSTALL_TORCH or not _torch_stack_ready(needs_cuda):
    target_label = "CUDA 12.1 (cu121)" if needs_cuda else "CPU"
    print(f"Instalando stack PyTorch para {target_label}...")
    _install_torch_stack(needs_cuda)
    print("Stack PyTorch instalada/atualizada.")
else:
    print("Stack PyTorch já compatível com este ambiente.")

required_packages = {
    "pycocotools": "pycocotools",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "Pillow": "PIL",
    "numpy": "numpy",
    "tqdm": "tqdm",
    "torchmetrics": "torchmetrics",
    "faster-coco-eval": "faster_coco_eval",
    "certifi": "certifi",
}

missing = [pkg for pkg, module in required_packages.items() if not _module_exists(module)]
if missing:
    _run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", *missing])
    print("Dependências instaladas:", ", ".join(missing))
else:
    print("Dependências já instaladas.")

import torch

print(f"Torch: {torch.__version__} | CUDA build: {torch.version.cuda} | cuda_available={torch.cuda.is_available()}")
if needs_cuda and not torch.cuda.is_available():
    print("ATENÇÃO: GPU NVIDIA detectada, mas CUDA indisponível. Reinicie o kernel e execute novamente esta célula.")

In [2]:
# Diagnóstico rápido do runtime PyTorch
import subprocess
import torch

def _has_nvidia_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "-L"],
            check=False,
            capture_output=True,
            text=True,
        )
        return result.returncode == 0 and bool(result.stdout.strip())
    except FileNotFoundError:
        return False

nvidia_gpu_detected = _has_nvidia_gpu()

print(f"GPU NVIDIA detectada: {nvidia_gpu_detected}")
print(f"Torch: {torch.__version__}")
print(f"CUDA build: {torch.version.cuda}")
print(f"cuda_available: {torch.cuda.is_available()}")

if nvidia_gpu_detected and not torch.cuda.is_available():
    print("ATENÇÃO: há GPU NVIDIA, porém CUDA não está ativa. Reexecute a célula 4 e reinicie o kernel.")
elif (not nvidia_gpu_detected) and torch.cuda.is_available():
    print("Observação: CUDA ativa, mas nvidia-smi não foi detectado no PATH.")
else:
    print("Ambiente de execução coerente para seguir com o notebook.")

GPU NVIDIA detectada: True
Torch: 2.5.1+cu121
CUDA build: 12.1
cuda_available: True
Ambiente de execução coerente para seguir com o notebook.


In [ ]:
import json
import random
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import tv_tensors
from torchvision.transforms import functional as F
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_V2_Weights,
    fasterrcnn_resnet50_fpn_v2,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from common_detection_protocol import (
    build_shared_train_transforms,
    evaluate_torchvision_coco,
)

sns.set_style("whitegrid")

In [ ]:
# Configuração de caminhos e experimento
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "pcb-defect-subset-5000"
RUNS_ROOT = PROJECT_ROOT / "runs" / "detect"
PROJECT_RUN_DIR = RUNS_ROOT / "tcc_pcb_defect_detection" / "fasterrcnn_resnet50_fpn_v2"
PROJECT_RUN_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DIR = DATASET_ROOT / "train" / "images"
VALID_DIR = DATASET_ROOT / "val" / "images"
TEST_DIR = DATASET_ROOT / "test" / "images"

TRAIN_ANN_PATH = TRAIN_DIR / "train_annotations.json"
VALID_ANN_PATH = VALID_DIR / "val_annotations.json"
TEST_ANN_PATH = TEST_DIR / "test_annotations.json"

for p in [TRAIN_ANN_PATH, VALID_ANN_PATH, TEST_ANN_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Arquivo de anotação não encontrado: {p}")

SEED = 42
EPOCHS = 100
BATCH_SIZE = 4
NUM_WORKERS = 0
LEARNING_RATE = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
LR_STEP_SIZE = 15
LR_GAMMA = 0.1
EARLY_STOP_PATIENCE = 10
IGNORE_CATEGORY_ID = 0
EVAL_ON_CPU_WHEN_MPS = True
MAP_BACKEND = "pycocotools"
RUN_TAG = "standardized_protocol"
MODEL_INPUT_SIZE = 640
TEST_SCORE_THRESHOLD = 0.25
MATCH_IOU_THRESHOLD = 0.5

if torch.cuda.is_available():
    TRAIN_DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    TRAIN_DEVICE = torch.device("mps")
else:
    TRAIN_DEVICE = torch.device("cpu")

if TRAIN_DEVICE.type == "mps" and EVAL_ON_CPU_WHEN_MPS:
    EVAL_DEVICE = torch.device("cpu")
else:
    EVAL_DEVICE = TRAIN_DEVICE

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

with open(TRAIN_ANN_PATH, "r", encoding="utf-8") as f:
    train_meta = json.load(f)

with open(VALID_ANN_PATH, "r", encoding="utf-8") as f:
    valid_meta = json.load(f)

def check_coco_filename_leakage(train_meta, valid_meta):
    train_files = {Path(img.get("file_name", "")).name for img in train_meta.get("images", [])}
    valid_files = {Path(img.get("file_name", "")).name for img in valid_meta.get("images", [])}

    exact_overlap = sorted(train_files.intersection(valid_files))

    train_prefixes = {name.split("_")[0] for name in train_files if "_" in name}
    valid_prefixes = {name.split("_")[0] for name in valid_files if "_" in name}
    prefix_overlap = sorted(train_prefixes.intersection(valid_prefixes))

    return {
        "train_count": len(train_files),
        "valid_count": len(valid_files),
        "exact_overlap": exact_overlap,
        "prefix_overlap": prefix_overlap,
    }

leakage_report = check_coco_filename_leakage(train_meta, valid_meta)
print(f"Total de arquivos de treino: {leakage_report['train_count']}")
print(f"Total de arquivos de validação: {leakage_report['valid_count']}")

if leakage_report["exact_overlap"]:
    print(f"ALERTA: {len(leakage_report['exact_overlap'])} arquivos idênticos entre treino e validação.")
    print("Exemplos:", leakage_report["exact_overlap"][:5])
else:
    print("OK: não há arquivos idênticos entre treino e validação.")

if leakage_report["prefix_overlap"]:
    print(f"ATENÇÃO: {len(leakage_report['prefix_overlap'])} prefixos aparecem em ambos os splits.")
    print("Exemplos:", leakage_report["prefix_overlap"][:5])
else:
    print("OK: não há sobreposição de prefixos entre treino e validação.")

all_categories = sorted(train_meta["categories"], key=lambda item: item["id"])
defect_categories = [item for item in all_categories if item["id"] != IGNORE_CATEGORY_ID]

cat_id_to_label = {item["id"]: idx + 1 for idx, item in enumerate(defect_categories)}
label_to_name = {idx + 1: item["name"] for idx, item in enumerate(defect_categories)}
label_to_cat_id = {idx + 1: item["id"] for idx, item in enumerate(defect_categories)}
NUM_CLASSES = len(label_to_name) + 1  # + background

class_df = pd.DataFrame(
    [
        {"label": label, "cat_id": label_to_cat_id[label], "class_name": label_to_name[label]}
        for label in sorted(label_to_name.keys())
    ]
)

print(f"Dataset base: {DATASET_ROOT}")
print(f"Diretório de saída: {PROJECT_RUN_DIR}")
print(f"Treino em: {TRAIN_DEVICE}")
print(f"Validação em: {EVAL_DEVICE}")
print(f"BATCH_SIZE: {BATCH_SIZE} | NUM_WORKERS: {NUM_WORKERS}")
print(f"LR: {LEARNING_RATE} | Early Stop Patience: {EARLY_STOP_PATIENCE}")
print(f"Categorias COCO totais: {len(all_categories)}")
print(f"Categorias de defeito utilizadas: {len(defect_categories)}")
display(class_df)

In [ ]:
class CocoDetectionDataset(Dataset):
    def __init__(self, images_dir, annotations_path, cat_id_to_label, train=False):
        self.images_dir = Path(images_dir)
        self.annotations_path = Path(annotations_path)
        self.cat_id_to_label = dict(cat_id_to_label)
        self.train = train
        self.transforms = build_shared_train_transforms() if train else None

        with open(self.annotations_path, "r", encoding="utf-8") as f:
            coco = json.load(f)

        self.images_meta = {img["id"]: img for img in coco["images"]}
        self.image_ids = sorted(self.images_meta.keys())

        self.annotations_by_image = defaultdict(list)
        skipped = 0

        for ann in coco["annotations"]:
            cat_id = ann.get("category_id")
            if cat_id not in self.cat_id_to_label:
                continue

            image_id = ann.get("image_id")
            if image_id not in self.images_meta:
                skipped += 1
                continue

            x, y, w, h = ann.get("bbox", [0, 0, 0, 0])
            if w <= 1 or h <= 1:
                skipped += 1
                continue

            width = self.images_meta[image_id]["width"]
            height = self.images_meta[image_id]["height"]

            x1 = max(0.0, float(x))
            y1 = max(0.0, float(y))
            x2 = min(float(width), float(x + w))
            y2 = min(float(height), float(y + h))

            if x2 <= x1 or y2 <= y1:
                skipped += 1
                continue

            self.annotations_by_image[image_id].append(
                {
                    "bbox": [x1, y1, x2, y2],
                    "label": int(self.cat_id_to_label[cat_id]),
                    "iscrowd": int(ann.get("iscrowd", 0)),
                }
            )

        self.skipped_annotations = skipped

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_meta = self.images_meta[image_id]
        image_path = self.images_dir / image_meta["file_name"]
        if not image_path.exists():
            raise FileNotFoundError(f"Imagem não encontrada: {image_path}")

        image = Image.open(image_path).convert("RGB")
        image = F.pil_to_tensor(image).float() / 255.0

        anns = self.annotations_by_image.get(image_id, [])
        if anns:
            boxes = torch.tensor([a["bbox"] for a in anns], dtype=torch.float32)
            labels = torch.tensor([a["label"] for a in anns], dtype=torch.int64)
            iscrowd = torch.tensor([a["iscrowd"] for a in anns], dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) if boxes.numel() > 0 else torch.zeros((0,), dtype=torch.float32)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([image_id], dtype=torch.int64),
            "area": area,
            "iscrowd": iscrowd,
        }

        if self.transforms is not None:
            image = tv_tensors.Image(image)
            target["boxes"] = tv_tensors.BoundingBoxes(
                target["boxes"],
                format="XYXY",
                canvas_size=(int(image.shape[-2]), int(image.shape[-1])),
            )
            image, target = self.transforms(image, target)
            image = image.as_subclass(torch.Tensor)
            transformed_boxes = target["boxes"].as_subclass(torch.Tensor)
            valid = (
                (transformed_boxes[:, 2] - transformed_boxes[:, 0] > 1.0)
                & (transformed_boxes[:, 3] - transformed_boxes[:, 1] > 1.0)
            )
            target["boxes"] = transformed_boxes[valid]
            target["labels"] = target["labels"][valid]
            target["iscrowd"] = target["iscrowd"][valid]
            target["area"] = (
                (target["boxes"][:, 2] - target["boxes"][:, 0])
                * (target["boxes"][:, 3] - target["boxes"][:, 1])
            )

        return image, target


def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


train_dataset = CocoDetectionDataset(
    images_dir=TRAIN_DIR,
    annotations_path=TRAIN_ANN_PATH,
    cat_id_to_label=cat_id_to_label,
    train=True,
)

valid_dataset = CocoDetectionDataset(
    images_dir=VALID_DIR,
    annotations_path=VALID_ANN_PATH,
    cat_id_to_label=cat_id_to_label,
    train=False,
)

test_dataset = CocoDetectionDataset(
    images_dir=TEST_DIR,
    annotations_path=TEST_ANN_PATH,
    cat_id_to_label=cat_id_to_label,
    train=False,
)

pin_memory = TRAIN_DEVICE.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    collate_fn=collate_fn,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    collate_fn=collate_fn,
)

print(f"Amostras de treino: {len(train_dataset)}")
print(f"Amostras de validação: {len(valid_dataset)}")
print(f"Amostras de teste: {len(test_dataset)}")
print(f"Anotações descartadas (treino): {train_dataset.skipped_annotations}")
print(f"Anotações descartadas (validação): {valid_dataset.skipped_annotations}")
print(f"Anotações descartadas (teste): {test_dataset.skipped_annotations}")

sample_images, sample_targets = next(iter(train_loader))
print(f"Batch de sanity check: {len(sample_images)} imagens")
print(f"Primeira imagem shape: {tuple(sample_images[0].shape)}")
print(f"Primeiro target - boxes: {sample_targets[0]['boxes'].shape}, labels: {sample_targets[0]['labels'].shape}")

## 3. Arquitetura do Modelo: Faster R-CNN

Neste estudo, o modelo **Faster R-CNN** é adotado como referência de detector em duas etapas para inspeção automática de PCBs, priorizando qualidade de localização para defeitos pequenos.

Diferente de detectores de estágio único, o Faster R-CNN primeiro propõe regiões candidatas e depois classifica/refina cada proposta. Esse fluxo tende a melhorar a precisão espacial em cenários com defeitos sutis.

### Por que Faster R-CNN para PCBs?
A escolha desta arquitetura considera três pontos relevantes para controle de qualidade industrial:

>* **Maior precisão de localização:** o mecanismo de propostas + refinamento ajuda a delimitar melhor defeitos pequenos como *missing_hole* e *spur*.
>* **Redução de falsos positivos:** a segunda etapa de classificação filtra melhor regiões ambíguas em placas com texturas complexas.
>* **Robustez para análise detalhada:** o pipeline em duas etapas favorece inspeções com maior exigência de qualidade de caixa.

### Estrutura Simplificada
O fluxo do Faster R-CNN pode ser resumido nas etapas abaixo:

>1. **Input:** imagem da PCB é processada para extração de características.
>2. **Backbone + FPN:** a rede extrai mapas multiescala (detalhe fino + contexto global).
>3. **RPN (Region Proposal Network):** gera regiões candidatas com maior probabilidade de conter defeitos.
>4. **RoI Align + Head:** recorta/normaliza as propostas e executa classificação + regressão de caixa.
>5. **Saídas finais:** classe do defeito e *bounding box* refinada.

<div align="center">
  <h3>Arquitetura Faster R-CNN</h3>
  <img src="https://miro.medium.com/1*3AZI4cNNOaLjuk0c73v7zg.jpeg" width="760" alt="Diagrama Faster R-CNN">
  <p><em>Figura 1: Fluxo geral do Faster R-CNN com backbone, RPN e etapa de refinamento por RoI.</em></p>
</div>

### Leitura prática da arquitetura
1. **Extração inicial de características:** o backbone transforma pixels em descritores visuais úteis para detecção.
2. **Proposição de regiões:** o RPN sugere onde pode existir defeito, reduzindo busca exaustiva.
3. **Refinamento final:** cada proposta é classificada e ajustada, melhorando precisão de localização e classe.

In [ ]:
# Inicializa o modelo com tentativa de pesos pré-treinados e fallback offline em caso de SSL
import os
import ssl

# Tenta normalizar CA bundle para permitir download HTTPS dos pesos
try:
    import certifi
    ca_bundle = certifi.where()
    os.environ["SSL_CERT_FILE"] = ca_bundle
    os.environ["REQUESTS_CA_BUNDLE"] = ca_bundle
    ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=ca_bundle)
except Exception:
    ca_bundle = None

weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
try:
    model = fasterrcnn_resnet50_fpn_v2(
        weights=weights,
        min_size=MODEL_INPUT_SIZE,
        max_size=MODEL_INPUT_SIZE,
        box_score_thresh=0.001,
        box_detections_per_img=300,
    )
except Exception as exc:
    raise RuntimeError(
        "Não foi possível carregar os pesos COCO do Faster R-CNN. "
        "O protocolo padronizado não permite fallback silencioso para treino do zero."
    ) from exc

model_init_mode = "coco_pretrained_resnet50_fpn_v2"

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(TRAIN_DEVICE)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

print(f"Modelo pronto com {NUM_CLASSES} classes totais (inclui background).")
print(f"Modo de inicialização: {model_init_mode}")
if ca_bundle is not None:
    print(f"CA bundle configurado: {ca_bundle}")
if ssl_error_message is not None:
    print(f"Erro original de SSL: {ssl_error_message}")

In [7]:
def move_targets_to_device(targets, device):
    moved = []
    for target in targets:
        moved.append({k: v.to(device) if torch.is_tensor(v) else v for k, v in target.items()})
    return moved


def train_one_epoch(model, data_loader, optimizer, device, epoch_index):
    model.train()

    running = defaultdict(float)
    seen = 0

    pbar = tqdm(data_loader, desc=f"Treinamento época {epoch_index:02d}", leave=False)

    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = move_targets_to_device(targets, device)

        loss_dict = model(images, targets)
        total_loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        batch_size = len(images)
        seen += batch_size

        running["loss"] += float(total_loss.item()) * batch_size
        for key, value in loss_dict.items():
            running[key] += float(value.item()) * batch_size

        pbar.set_postfix({"loss": f"{total_loss.item():.4f}"})

    return {key: value / max(1, seen) for key, value in running.items()}


def build_map_metric(backend="faster_coco_eval"):
    try:
        return MeanAveragePrecision(iou_type="bbox", backend=backend)
    except Exception as exc:
        print(f"Backend {backend} indisponível ({exc}). Fallback para pycocotools.")
        return MeanAveragePrecision(iou_type="bbox")


def evaluate_map(model, data_loader, eval_device, backend="faster_coco_eval"):
    original_device = next(model.parameters()).device
    need_restore = eval_device != original_device

    if need_restore:
        model.to(eval_device)

    model.eval()
    metric = build_map_metric(backend=backend)

    with torch.no_grad():
        pbar = tqdm(data_loader, desc="Validação mAP", leave=False)
        for images, targets in pbar:
            images_eval = [img.to(eval_device) for img in images]
            outputs = model(images_eval)

            preds = []
            refs = []

            for output, target in zip(outputs, targets):
                preds.append(
                    {
                        "boxes": output["boxes"].detach().cpu(),
                        "scores": output["scores"].detach().cpu(),
                        "labels": output["labels"].detach().cpu(),
                    }
                )

                refs.append(
                    {
                        "boxes": target["boxes"].detach().cpu(),
                        "labels": target["labels"].detach().cpu(),
                    }
                )

            metric.update(preds, refs)

    raw = metric.compute()

    model.train()
    if need_restore:
        model.to(original_device)

    scalar_keys = ["map", "map_50", "map_75", "mar_1", "mar_10", "mar_100"]
    results = {}

    for key in scalar_keys:
        if key in raw and torch.is_tensor(raw[key]) and raw[key].numel() == 1:
            results[key] = float(raw[key].item())

    return results

In [ ]:
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M")
run_name = f"{run_timestamp}_fasterrcnn_resnet50_fpn_v2_e{EPOCHS}_bs{BATCH_SIZE}_seed{SEED}_{RUN_TAG}"
run_dir = PROJECT_RUN_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=True)

best_ckpt_path = run_dir / "best_model.pth"
last_ckpt_path = run_dir / "last_model.pth"
history_csv_path = run_dir / "training_history.csv"
history_json_path = run_dir / "training_history.json"
history_plot_path = run_dir / "training_curves.png"

config_path = run_dir / "config.json"
class_map_path = run_dir / "class_map.json"

config_data = {
    "run_name": run_name,
    "run_tag": RUN_TAG,
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "model_input_size": MODEL_INPUT_SIZE,
    "coco_prediction_score_floor": 0.001,
    "max_detections_per_image": 300,
    "augmentation_policy": "shared_moderate_v1",
    "num_workers": NUM_WORKERS,
    "learning_rate": LEARNING_RATE,
    "momentum": MOMENTUM,
    "weight_decay": WEIGHT_DECAY,
    "lr_step_size": LR_STEP_SIZE,
    "lr_gamma": LR_GAMMA,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "ignore_category_id": IGNORE_CATEGORY_ID,
    "train_device": str(TRAIN_DEVICE),
    "eval_device": str(EVAL_DEVICE),
    "map_backend": MAP_BACKEND,
    "num_classes_total": NUM_CLASSES,
    "test_score_threshold": TEST_SCORE_THRESHOLD,
    "match_iou_threshold": MATCH_IOU_THRESHOLD,
    "dataset_root": str(DATASET_ROOT),
    "train_annotation_path": str(TRAIN_ANN_PATH),
    "valid_annotation_path": str(VALID_ANN_PATH),
    "test_annotation_path": str(TEST_ANN_PATH),
    "model_architecture": "Faster R-CNN ResNet-50-FPN V2",
    "optimizer": "SGD",
    "loss_function": "Cross-Entropy + Smooth L1",
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_data, f, indent=2)

with open(class_map_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "label_to_name": {str(k): v for k, v in label_to_name.items()},
            "label_to_cat_id": {str(k): int(v) for k, v in label_to_cat_id.items()},
        },
        f,
        indent=2,
    )

print(f"Run atual: {run_dir}")
print("✓ Configuração salva. Dataset deve ser carregado da célula anterior.")
print("✓ Treinamento está pronto para ser executado com train_loader e valid_loader.")
print(f"✓ Checkpoints serão salvos em: {run_dir}")

history = []
best_map = -1.0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_stats = train_one_epoch(model, train_loader, optimizer, TRAIN_DEVICE, epoch)
    val_stats = evaluate_map(model, valid_loader, EVAL_DEVICE, backend=MAP_BACKEND)

    lr_scheduler.step()
    current_lr = float(optimizer.param_groups[0]["lr"])

    row = {
        "epoch": epoch,
        "lr": current_lr,
        **{f"train_{k}": float(v) for k, v in train_stats.items()},
        **{f"val_{k}": float(v) for k, v in val_stats.items()},
    }
    history.append(row)

    current_map = row.get("val_map", -1.0)
    if np.isnan(current_map):
        current_map = -1.0

    is_best = current_map > best_map
    if is_best:
        best_map = current_map
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    checkpoint_data = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "lr_scheduler_state_dict": lr_scheduler.state_dict(),
        "best_map": best_map,
        "config": config_data,
        "label_to_name": label_to_name,
        "label_to_cat_id": label_to_cat_id,
    }

    torch.save(checkpoint_data, last_ckpt_path)
    if is_best:
        torch.save(checkpoint_data, best_ckpt_path)

    print(
        f"Época {epoch:02d}/{EPOCHS} | "
        f"train_loss={row.get('train_loss', float('nan')):.4f} | "
        f"val_map={row.get('val_map', float('nan')):.4f} | "
        f"val_map50={row.get('val_map_50', float('nan')):.4f} | "
        f"patience={epochs_without_improvement}/{EARLY_STOP_PATIENCE}"
    )

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f"Early stopping acionado após {epoch} épocas sem melhora em val_map.")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(history_csv_path, index=False)
history_df.to_json(history_json_path, orient="records", indent=2)

print(f"Histórico salvo em: {history_csv_path}")
print(f"Melhor checkpoint: {best_ckpt_path}")
print(f"Último checkpoint: {last_ckpt_path}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
axes[0].set_title("Evolução da perda de treino")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].legend()

if "val_map" in history_df.columns:
    axes[1].plot(history_df["epoch"], history_df["val_map"], marker="o", label="mAP")
if "val_map_50" in history_df.columns:
    axes[1].plot(history_df["epoch"], history_df["val_map_50"], marker="o", label="mAP50")

axes[1].set_title("Evolução de mAP na validação")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Score")
if axes[1].lines:
    axes[1].legend()

plt.tight_layout()
plt.savefig(history_plot_path, dpi=180)
plt.show()
plt.close(fig)

print(f"Curvas salvas em: {history_plot_path}")

## 4. Análise de Métricas de Desempenho

Para validar a eficácia do Faster R-CNN na detecção de defeitos em PCBs, utilizamos um conjunto de métricas alinhado ao padrão adotado no notebook do YOLO, com foco em interpretação prática para inspeção industrial.

### Matriz de Confusão
A matriz de confusão mostra, classe por classe, quais defeitos foram corretamente identificados e onde ocorreram confusões.

- **Impacto industrial:** ajuda a identificar erros críticos, como falsos negativos em defeitos que não podem escapar da inspeção.

### Precisão (Precision)
A precisão responde: **"de todos os defeitos que o modelo apontou, quantos eram reais?"**

$$\text{Precision} = \frac{\text{Verdadeiros Positivos}}{\text{Verdadeiros Positivos} + \text{Falsos Positivos}}$$

### Recall (Sensibilidade)
O recall responde: **"de todos os defeitos existentes, quantos o modelo encontrou?"**

$$\text{Recall} = \frac{\text{Verdadeiros Positivos}}{\text{Verdadeiros Positivos} + \text{Falsos Negativos}}$$

### F1-Score
O F1-Score equilibra precisão e recall em um único indicador.

$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

### mAP (Mean Average Precision)
- **mAP@50:** indica a qualidade geral de detecção com critério IoU mais permissivo.
- **mAP@50-95:** métrica mais rigorosa, útil para avaliar a qualidade fina de localização das caixas.

Nesta seção, além das curvas de treino, será gerado um resumo final em JSON e visualizações complementares para apoiar a análise após o último treinamento.

In [ ]:
if not best_ckpt_path.exists() and not last_ckpt_path.exists():
    raise FileNotFoundError("Nenhum checkpoint encontrado para avaliação final.")

checkpoint_path = best_ckpt_path if best_ckpt_path.exists() else last_ckpt_path
checkpoint_data = torch.load(checkpoint_path, map_location="cpu")
model.load_state_dict(checkpoint_data["model_state_dict"])
model.to(TRAIN_DEVICE)
print(f"Checkpoint carregado para avaliação final: {checkpoint_path}")

metrics_summary = evaluate_torchvision_coco(
    model=model,
    data_loader=test_loader,
    annotation_path=TEST_ANN_PATH,
    label_to_category_id=label_to_cat_id,
    device=EVAL_DEVICE,
    output_json_path=run_dir / "coco_test_predictions.json",
)
metrics_summary.update(
    {
        "run_name": run_name,
        "run_dir": str(run_dir),
        "checkpoint": str(checkpoint_path),
        "evaluation_backend": "pycocotools.COCOeval",
    }
)

metrics_summary_path = run_dir / "metrics_summary.json"
with open(metrics_summary_path, "w", encoding="utf-8") as file:
    json.dump(metrics_summary, file, indent=2)

print("\n--- Métricas comuns de teste ---")
print(f"mAP@50: {metrics_summary['map50']:.4f}")
print(f"mAP@50-95: {metrics_summary['map50_95']:.4f}")
print(f"Precision macro: {metrics_summary['precision_macro']:.4f}")
print(f"Recall macro: {metrics_summary['recall_macro']:.4f}")
print(f"F1 macro: {metrics_summary['f1_macro']:.4f}")
print(f"Resumo salvo em: {metrics_summary_path}")

metrics_by_class_df = pd.DataFrame(metrics_summary["per_class"])
display(metrics_by_class_df)

positions = np.arange(len(metrics_by_class_df))
width = 0.25
plt.figure(figsize=(12, 5))
plt.bar(positions - width, metrics_by_class_df["precision"], width=width, label="Precision")
plt.bar(positions, metrics_by_class_df["recall"], width=width, label="Recall")
plt.bar(positions + width, metrics_by_class_df["f1"], width=width, label="F1")
plt.xticks(positions, metrics_by_class_df["class_name"], rotation=30, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Métricas por classe no conjunto de teste")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Testes de Inferência

A etapa de inferência foi separada para o notebook `FasterRCNN_inferencia.ipynb`, mantendo este arquivo focado em preparação, treinamento e análise de métricas.

Após finalizar um novo treinamento, utilize o melhor checkpoint (`best_model.pth`) da execução mais recente para validar o comportamento em imagens de teste e em imagens externas.

## 6. Conclusão

O notebook está configurado para retreinar o **Faster R-CNN** com entrada interna de 640 × 640 pixels, augmentation moderado compartilhado, limite máximo de 100 épocas e early stopping baseado no mAP@50:95 de validação. O modelo mantém os pesos COCO da variante ResNet-50-FPN V2 e substitui a cabeça de classificação.

A avaliação final utiliza o mesmo backend `pycocotools.COCOeval` empregado nos notebooks da Ultralytics. Os resultados numéricos anteriores foram removidos desta conclusão e devem ser preenchidos somente após a execução integral do novo treinamento.

## 7. Referências

> Documentação TorchVision (Detecção): https://pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn_v2.html

> Documentação PyTorch: https://docs.pytorch.org/docs/stable/index.html

> TorchMetrics - MeanAveragePrecision: https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision.html

> Stanford CNN Cheatsheet: https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-convolutional-neural-networks

> Materiais de Aula